# Docker

Lý thuyết
* Container
* Image
* Layer và copy-on-write
* Registry
* Docker Engine
* Namespace và cgroups
* Port mapping
* Volume và bind mount
* Docker network

Input: 
1. Source code
2. Dockerfile
3. Dependencies
4. Config / environment variables
5. Port mapping
6. Volume

Pipeline: 
* Build pipeline: quá trình biến source code + Dockerfile thành image.
* Run pipeline: quá trình chạy image để tạo container.

Output: 
* Image
* Container
* log
* API response
* Dữ liệu trong volume


## 1. Input

### 1.1. Source code

Ví dụ project FastAPI:

```text
project/
├── main.py
├── requirements.txt
└── Dockerfile
```

Source code là phần ứng dụng bạn muốn đóng gói vào Docker.

---

### 1.2. Dockerfile

`Dockerfile` là file hướng dẫn Docker cách tạo image.

Ví dụ:

```dockerfile
FROM python:3.11

WORKDIR /app

COPY requirements.txt .
RUN pip install -r requirements.txt

COPY . .

CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]
```

Ý nghĩa:

```text
1. Dùng Python 3.11 làm môi trường nền
2. Tạo thư mục /app trong container
3. Copy requirements.txt vào container
4. Cài thư viện Python
5. Copy toàn bộ source code vào container
6. Khi container chạy thì khởi động server FastAPI
```

---

### 1.3. Dependencies

Dependencies là các thư viện mà ứng dụng cần.

Ví dụ `requirements.txt`:

```txt
fastapi
uvicorn
qdrant-client
```

Docker sẽ dùng file này để cài thư viện vào image.

---

### 1.4. Environment variables

Environment variables là các biến cấu hình truyền vào container.

Ví dụ:

```bash
docker run -e DATABASE_URL=postgresql://user:pass@db:5432/app my-server
```

Trong đó:

```text
DATABASE_URL = thông tin kết nối database
```

---

### 1.5. Port mapping

Port mapping giúp máy thật truy cập được server chạy bên trong container.

Ví dụ:

```bash
docker run -p 8000:8000 my-server
```

Ý nghĩa:

```text
port 8000 trên máy thật → port 8000 trong container
```

Khi bạn mở:

```text
http://localhost:8000
```

request sẽ được chuyển vào server trong container.

---

### 1.6. Volume

Volume dùng để lưu dữ liệu lâu dài hoặc chia sẻ dữ liệu giữa máy thật và container.

Ví dụ:

```bash
docker run -v qdrant_storage:/qdrant/storage qdrant/qdrant
```

Ý nghĩa:

```text
Dữ liệu Qdrant sẽ được lưu trong volume qdrant_storage
```

Nếu container bị xóa, dữ liệu trong volume vẫn có thể còn.

---

## 2. Pipeline của Docker

Docker thường có 2 pipeline chính:

```text
Build pipeline
Run pipeline
```

---

### 2.1. Build pipeline

Build pipeline là quá trình biến source code + Dockerfile thành Docker image.

```text
Source code + Dockerfile + Dependencies
        ↓
docker build
        ↓
Docker image
```

Ví dụ:

```bash
docker build -t my-fastapi-app .
```

Giải thích:

```text
docker build       tạo image
-t my-fastapi-app  đặt tên image
.                  dùng Dockerfile trong thư mục hiện tại
```

Sau khi build xong, kiểm tra image:

```bash
docker images
```

Ví dụ kết quả:

```text
REPOSITORY          TAG       IMAGE ID
my-fastapi-app      latest    abc123
```

Image giống như một bản đóng gói hoàn chỉnh của ứng dụng.

---

### 2.2. Run pipeline

Run pipeline là quá trình chạy image để tạo container.

```text
Docker image
    ↓
docker run
    ↓
Container
    ↓
Server / app đang chạy
```

Ví dụ:

```bash
docker run --name fastapi-server -p 8000:8000 my-fastapi-app
```

Ý nghĩa:

```text
--name fastapi-server  đặt tên container
-p 8000:8000           map port từ máy thật vào container
my-fastapi-app         image được dùng để tạo container
```

---

## 3. Output của Docker là gì?

Output của Docker có thể là:

```text
1. Docker image
2. Container
3. Logs
4. API response
5. Files
6. Database data trong volume
```

---

### 3.1. Output là image

Sau lệnh:

```bash
docker build -t my-app .
```

Output là:

```text
Docker image: my-app
```

Image chưa phải app đang chạy. Nó là bản đóng gói để tạo container.

---

### 3.2. Output là container

Sau lệnh:

```bash
docker run my-app
```

Output là:

```text
Container đang chạy
```

Container là ứng dụng thật sự đang chạy.

---

### 3.3. Output là log

Xem log container:

```bash
docker logs fastapi-server
```

Ví dụ output:

```text
INFO: Uvicorn running on http://0.0.0.0:8000
```

Log giúp kiểm tra server có chạy đúng hay bị lỗi.

---

### 3.4. Output là API response

Nếu container chạy server, bạn có thể gọi:

```text
http://localhost:8000
```

Ví dụ output:

```json
{
  "message": "Hello from Docker"
}
```

---

### 3.5. Output là dữ liệu trong volume

Ví dụ PostgreSQL, Qdrant, MongoDB chạy trong Docker thì dữ liệu thường được lưu trong volume.

```text
Container PostgreSQL
        ↓
Volume
        ↓
Database data
```

---

## 4. Sơ đồ tổng quát

```text
INPUT
│
├── Source code
├── Dockerfile
├── requirements.txt / package.json
├── Environment variables
├── Port mapping
└── Volume
        ↓
DOCKER BUILD
        ↓
IMAGE
        ↓
DOCKER RUN
        ↓
CONTAINER
        ↓
OUTPUT
│
├── Running server
├── Logs
├── API response
├── Files
└── Database data in volume
```

---

## 5. Ví dụ thực tế với FastAPI

### Input

```text
main.py
requirements.txt
Dockerfile
```

---

### File `main.py`

```python
from fastapi import FastAPI

app = FastAPI()

@app.get("/")
def home():
    return {"message": "Server is running in Docker"}
```

---

### File `requirements.txt`

```txt
fastapi
uvicorn
```

---

### File `Dockerfile`

```dockerfile
FROM python:3.11

WORKDIR /app

COPY requirements.txt .
RUN pip install -r requirements.txt

COPY . .

CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]
```

---

### Build image

```bash
docker build -t fastapi-demo .
```

Output:

```text
image: fastapi-demo
```

---

### Run container

```bash
docker run --name my-api -p 8000:8000 fastapi-demo
```

Output:

```text
container: my-api
server chạy ở localhost:8000
```

---

### Gọi API

Mở trình duyệt:

```text
http://localhost:8000
```

Output:

```json
{
  "message": "Server is running in Docker"
}
```


# Code

In [11]:
!docker --version

Docker version 29.4.1, build 055a478


In [12]:
!docker info

Client:
 Version:    29.4.1
 Context:    desktop-linux
 Debug Mode: true
 Plugins:
  agent: Docker AI Agent Runner (Docker Inc.)
    Version:  v1.44.0
    Path:     C:\Program Files\Docker\cli-plugins\docker-agent.exe
  ai: Docker AI Agent - Ask Gordon (Docker Inc.)
    Version:  v1.20.2
    Path:     C:\Program Files\Docker\cli-plugins\docker-ai.exe
  buildx: Docker Buildx (Docker Inc.)
    Version:  v0.33.0-desktop.1
    Path:     C:\Program Files\Docker\cli-plugins\docker-buildx.exe
  compose: Docker Compose (Docker Inc.)
    Version:  v5.1.3
    Path:     C:\Program Files\Docker\cli-plugins\docker-compose.exe
  debug: Get a shell into any image or container (Docker Inc.)
    Version:  0.0.47
    Path:     C:\Program Files\Docker\cli-plugins\docker-debug.exe
  desktop: Docker Desktop commands (Docker Inc.)
    Version:  v0.3.0
    Path:     C:\Program Files\Docker\cli-plugins\docker-desktop.exe
  dhi: CLI for managing Docker Hardened Images (Docker Inc.)
    Version:  v0.0.2
    Pat

### 1. Image

In [13]:
!docker images

IMAGE                                                  ID             DISK USAGE   CONTENT SIZE   EXTRA
docker.elastic.co/elasticsearch/elasticsearch:8.15.5   8870d5b7b862       1.96GB          673MB        
mailhog/mailhog:latest                                 8d76a3d4ffa3        572MB          146MB        
milvusdb/milvus:v2.5.10                                02e1d60d71ab       2.41GB          676MB        
milvusdb/milvus:v3.0-beta                              0a030f267a16       3.75GB         1.02GB        
minio/minio:RELEASE.2024-12-18T13-15-44Z               1dce27c494a1        244MB         62.6MB        
minio/minio:RELEASE.2025-04-22T22-12-26Z               a1ea29fa2835        250MB           64MB        
minio/minio:latest                                     14cea493d9a3        241MB         62.2MB        
multimodal-retrieval-backend:latest                    73602cae41bd        898MB          212MB        
multimodal-retrieval-web:latest                        425a36562

In [14]:
# Pull image from Docker Hub
!docker pull qdrant/qdrant

Using default tag: latest
latest: Pulling from qdrant/qdrant
4f4fb700ef54: Pulling fs layer
4f4fb700ef54: Pulling fs layer
4f4fb700ef54: Pulling fs layer
c57b3684dca6: Pulling fs layer
ea89e0657a56: Pulling fs layer
4f4fb700ef54: Pulling fs layer
c19de56e02d2: Pulling fs layer
169e35931ada: Pulling fs layer
4f4fb700ef54: Pulling fs layer
63c3693beb8f: Pulling fs layer
a621d80a8a37: Pulling fs layer
4f4fb700ef54: Pulling fs layer
4f4fb700ef54: Pulling fs layer
4f4fb700ef54: Already exists
4f4fb700ef54: Pull complete
a621d80a8a37: Download complete
c19de56e02d2: Download complete
ea89e0657a56: Download complete
169e35931ada: Download complete
c19de56e02d2: Pull complete
391e68678c5c: Download complete
f08b302999db: Download complete
c57b3684dca6: Download complete
63c3693beb8f: Download complete
169e35931ada: Pull complete
c57b3684dca6: Pull complete
a621d80a8a37: Pull complete
ea89e0657a56: Pull complete
63c3693beb8f: Pull complete
Digest: sha256:75eab8c4ba42096724fdcfde8b4de0b5713d529d

In [ ]:
# Remove image from local registry
# docker rmi my-api:1.0

### 2. Container

In [ ]:
# List all containers running
!docker ps

CONTAINER ID   IMAGE     COMMAND                  CREATED         STATUS         PORTS                                     NAMES
6233857a5e03   nginx     "/docker-entrypoint.…"   4 minutes ago   Up 4 minutes   0.0.0.0:8080->80/tcp, [::]:8080->80/tcp   web


2. Xem tất cả container, kể cả đã dừng

In [22]:
# List all containers running and stopped
!docker ps -a 

CONTAINER ID   IMAGE                COMMAND                  CREATED          STATUS                       PORTS                                     NAMES
6233857a5e03   nginx                "/docker-entrypoint.…"   5 minutes ago    Up 5 minutes                 0.0.0.0:8080->80/tcp, [::]:8080->80/tcp   web
659e882f4b87   b3063c673f39         "./entrypoint.sh"        46 minutes ago   Exited (143) 3 minutes ago                                             funny_wright
759185d6bc94   zilliz/attu:latest   "docker-entrypoint.s…"   13 days ago      Exited (137) 12 days ago                                               jovial_gates
7ef1559c2246   b3063c673f39         "./entrypoint.sh"        13 days ago      Exited (143) 13 days ago                                               qdrant


In [10]:
!docker run qdrant/qdrant

^C


           _                 _    
  __ _  __| |_ __ __ _ _ __ | |_  
 / _` |/ _` | '__/ _` | '_ \| __| 
| (_| | (_| | | | (_| | | | | |_  
 \__, |\__,_|_|  \__,_|_| |_|\__| 
    |_|                           

Version: 1.18.0, build: db3fca32
Access web UI at http://localhost:6333/dashboard
Agentic Skills: https://skills.qdrant.tech/

2026-06-09T04:21:02.082995Z  WARN qdrant: There is a potential issue with the filesystem for storage path ./storage. Details: Container filesystem detected - storage might be lost with container re-creation
2026-06-09T04:21:02.084327Z  INFO storage::content_manager::consensus::persistent: Initializing new raft state at ./storage/raft_state.json
2026-06-09T04:21:02.200291Z  INFO qdrant: Distributed mode disabled
2026-06-09T04:21:02.200648Z  INFO qdrant: Telemetry reporting enabled, id: 9eaf9798-1a08-4a8d-9ebe-e563d4e715fe
2026-06-09T04:21:02.378027Z  INFO qdrant::tonic: Qdrant gRPC listening on 6334
2026-06-09T04:21:02.378062Z  INFO qdrant::tonic: TLS dis

In [19]:
# Stop the container
!docker stop 659e882f4b87

659e882f4b87


In [15]:
!docker run -d --name web -p 8080:80 nginx

6233857a5e03028bb93b4810514a4810afef9011dbd31a574dea11771dd5cb7f


Unable to find image 'nginx:latest' locally
latest: Pulling from library/nginx
830625e1ac85: Pulling fs layer
13fd728be9eb: Pulling fs layer
5431d0092ffd: Pulling fs layer
b4a248c845e5: Pulling fs layer
7f8b1a2b17d8: Pulling fs layer
45381ecb0e2f: Pulling fs layer
45381ecb0e2f: Download complete
13fd728be9eb: Download complete
b4a248c845e5: Download complete
7f8b1a2b17d8: Download complete
5431d0092ffd: Download complete
cc5f57206478: Download complete
5e6b66b5e5f1: Download complete
830625e1ac85: Download complete
45381ecb0e2f: Pull complete
b4a248c845e5: Pull complete
7f8b1a2b17d8: Pull complete
5431d0092ffd: Pull complete
830625e1ac85: Pull complete
13fd728be9eb: Pull complete
Digest: sha256:5aca99593157f4ae539a5dec1092a0ad8762f8e2eb1789085a13a0f5622369f6
Status: Downloaded newer image for nginx:latest


### 3. Log and debug 

In [20]:
!docker logs web

/docker-entrypoint.sh: /docker-entrypoint.d/ is not empty, will attempt to perform configuration
/docker-entrypoint.sh: Looking for shell scripts in /docker-entrypoint.d/
/docker-entrypoint.sh: Launching /docker-entrypoint.d/10-listen-on-ipv6-by-default.sh
10-listen-on-ipv6-by-default.sh: info: Getting the checksum of /etc/nginx/conf.d/default.conf
10-listen-on-ipv6-by-default.sh: info: Enabled listen on IPv6 in /etc/nginx/conf.d/default.conf
/docker-entrypoint.sh: Sourcing /docker-entrypoint.d/15-local-resolvers.envsh
/docker-entrypoint.sh: Launching /docker-entrypoint.d/20-envsubst-on-templates.sh
/docker-entrypoint.sh: Launching /docker-entrypoint.d/30-tune-worker-processes.sh
/docker-entrypoint.sh: Configuration complete; ready for start up


2026/06/09 05:01:55 [notice] 1#1: using the "epoll" event method
2026/06/09 05:01:55 [notice] 1#1: nginx/1.31.1
2026/06/09 05:01:55 [notice] 1#1: built by gcc 14.2.0 (Debian 14.2.0-19) 
2026/06/09 05:01:55 [notice] 1#1: OS: Linux 5.15.167.4-microsoft-standard-WSL2
2026/06/09 05:01:55 [notice] 1#1: getrlimit(RLIMIT_NOFILE): 1048576:1048576
2026/06/09 05:01:55 [notice] 1#1: start worker processes
2026/06/09 05:01:55 [notice] 1#1: start worker process 29
2026/06/09 05:01:55 [notice] 1#1: start worker process 30
2026/06/09 05:01:55 [notice] 1#1: start worker process 31
2026/06/09 05:01:55 [notice] 1#1: start worker process 32
2026/06/09 05:01:55 [notice] 1#1: start worker process 33
2026/06/09 05:01:55 [notice] 1#1: start worker process 34
2026/06/09 05:01:55 [notice] 1#1: start worker process 35
2026/06/09 05:01:55 [notice] 1#1: start worker process 36
2026/06/09 05:01:55 [notice] 1#1: start worker process 37
2026/06/09 05:01:55 [notice] 1#1: start worker process 38
2026/06/09 05:01:55 [n